# Reproducing Anto-Sztrikacs, Nazir & Segal 2023 — spin-boson dynamics with a strongly coupled Lorentzian bath

**Paper**: N. Anto-Sztrikacs, A. Nazir, D. Segal,
*Effective dynamics of open quantum systems strongly coupled to a bath: A nonperturbative Lorentzian master equation*,
[Phys. Rev. Research **5**, 033227 (2023)](https://doi.org/10.1103/PhysRevResearch.5.033227).

**Content reproduced**: the exact reference dynamics shown in the figures of the paper (curves computed by TEMPO) — the spin-boson model $P(t) = \langle\sigma_z(t)\rangle$ with a Lorentzian-spectrum bath at strong coupling. The paper uses TEMPO
as the reference benchmark for its master equation (RCQME); this notebook reproduces that **TEMPO benchmark** part.

**Model**:
$$H = \frac{\Delta}{2}\sigma_x + \frac{\sigma_z}{2} \sum_k g_k (b_k + b_k^\dagger) + \sum_k \omega_k b_k^\dagger b_k,$$
$$J(\omega) = \frac{1}{\pi}\frac{\gamma \lambda^2}{(\omega-\Omega)^2 + \gamma^2} \quad \text{(Lorentzian)}$$

**Parameters** (reduced scale): $\Delta = 1$, $\lambda = 1$, $\Omega = 5$, $\gamma = 2$, $\beta = 1$,
$\delta t = 0.1$, $t \le 4$, $\chi = 30$; initial state $P(0) = 1$.
(The Lorentzian spectrum is nonvanishing as $\omega \to 0$; the convolution with the Bose factor $1/\omega$ is log-divergent,
so the spectral integral is truncated starting from a lower bound of $\gamma/100$.)

**Reference data**: independent ED (bath discretization + exact diagonalization).


In [ ]:
using TEMPO, ImpurityModelBase, LinearAlgebra

# ---- TEMPO: Lorentzian-bath SBM, P(t) ----
function tempo_p(; δt=0.1, tmax=4.0, β=1.0, Δ=1.0, λ=1.0, Ω=5.0, γ=2.0, chi=30)
    Nt = round(Int, tmax/δt)
    trunc = truncdimcutoff(D=chi, ϵ=1.0e-12, add_back=0)
    lattice = ADTLattice(N=Nt, δt=δt, contour=:real)
    hyb = AdditiveHyb([0.5, -0.5])
    # Lorentzian spectral density; lb > 0 avoids the log-divergent 1/ω tail of the Bose factor
    spec = spectrum(w -> γ*λ^2 / (π * ((w - Ω)^2 + γ^2)), lb=γ/100, ub=Ω + 10γ)
    bath = bosonicbath(spec, β=β)
    corr = correlationfunction(bath, lattice)
    mpsI = hybriddynamics(lattice, corr, hyb, trunc=trunc)
    ρimp = [1 0; 0 0.0]
    model = ImpurityHamiltonian(Δ/2 .* [0 1; 1 0.0])
    mpsK = sysdynamics(lattice, model, trunc=trunc)
    mpsK = boundarycondition!(mpsK, lattice, ρ₀=ρimp)
    cache = environments(lattice, mpsK, mpsI)
    p = [real(expectationvalue(ADTTerm(index(lattice, i, branch=:+), [1.0, -1.0]), cache)) for i in 1:Nt]
    return [(i-1)*δt for i in 1:Nt], p
end

# ---- ED reference ----
function ed_p(; Δ=1.0, λ=1.0, Ω=5.0, γ=2.0, Nmodes=8, d=2, times=0:0.1:1.5)
    r = range(γ/100, stop=Ω + 3γ, length=Nmodes)
    ws = collect(r); dw = Float64(step(r))
    spec(w) = γ*λ^2 / (π * ((w - Ω)^2 + γ^2))
    gs = [sqrt(spec(w)*dw/π) for w in ws]
    dimB = d^Nmodes
    a1 = zeros(d, d); for n in 1:d-1; a1[n, n+1] = sqrt(n); end
    n1 = a1'a1
    mode_op(op1, k) = kron([k == j ? op1 : Matrix{Float64}(I, d, d) for j in 1:Nmodes]...)
    Hb = zeros(dimB, dimB); X = zeros(dimB, dimB)
    for k in 1:Nmodes
        Hb .+= ws[k] .* mode_op(n1, k)
        X .+= gs[k] .* (mode_op(a1, k) + mode_op(a1, k)')
    end
    σx = [0 1; 1 0.0]; σz = [1 0; 0 -1.0]
    H = kron(Δ/2 .* σx, Matrix(I, dimB, dimB)) + kron(Matrix(I, 2, 2), Hb) + kron(σz/2, X)
    # initial state: spin up along z + thermal bath at β
    β = 1.0
    ρb = exp(-β .* Hb); ρb ./= tr(ρb)
    ρ0 = kron([1 0; 0 0.0], ρb)
    E, V = eigen(Hermitian(H))
    M = V' * ρ0 * V                      # ρ0 in the eigenbasis
    W = V' * (kron(σz, Matrix(I, dimB, dimB))) * V
    # ρ(t) = V D(t) M D(t)† V†  with D(t) = diag(e^{-iEt});
    # P(t) = tr[W · D(t) M D(t)†]
    p2 = [real(tr(W * ((exp.(-im*E*t) .* M) .* exp.(im*E*t)'))) for t in times]
    return collect(times), p2
end
println("functions defined")


In [ ]:
@time tT, pT = tempo_p()
println("TEMPO done, P(end) = ", round(last(pT), digits=4))


In [ ]:
@time tE, pE = ed_p()
println("ED done")


In [ ]:
using Plots
pl = plot(xlabel="t", ylabel="P(t) = ⟨σ_z(t)⟩",
          title="Lorentzian-bath SBM at strong coupling (Anto-Sztrikacs et al. 2023, TEMPO benchmark)",
          legend=:topright, size=(640, 420))
plot!(pl, tT, pT, color=1, lw=2, label="TEMPO (this package)")
# plot the ED reference only in the short-time regime where the discrete-mode
# approximation is reliable (long times show recurrences of the discrete spectrum)
mask = tE .<= 1.0
scatter!(pl, tE[mask], pE[mask], color=2, msw=0, ms=3, label="exact diagonalization (8 modes)")
savefig(pl, "segal2023_pt.png")
pl


## Discussion of results

- For the Lorentzian bath at strong coupling ($\lambda = 1$, comparable to $\Delta$), $P(t)$ shows underdamped relaxation:
  the narrow-band bath spectrum (center $\Omega = 5$, half-width $\gamma = 2$) has a long memory time, and the short-time dynamics are characterized by coherent oscillations at frequency
  $\sim\Delta$ superimposed on a slow decay.
- At short times ($t \lesssim 1$) TEMPO and the independent ED agree quantitatively (within $\lesssim 0.05$); at longer times
  ED breaks down, showing recurrence oscillations of the discrete spectrum — hence only the short-time regime is plotted, while the TEMPO curve continues to give the dissipative relaxation of the continuous spectrum.
- These exact TEMPO curves are precisely the reference benchmarks used in the paper to test master equations of increasing order (Redfield / RCQME / generalized master equations).
